In [ ]:
%run Utils_Log
setup_log("silver_current_year")

In [ ]:
from pyspark.sql.functions import col, lit, regexp_replace, when, round, trim, expr, broadcast, to_date, coalesce, isnull, count, month
from pyspark.sql.types import FloatType, IntegerType

log("Extracción: leyendo Files/Bronze/Ventas/anio=2026/pescafresca_2026.csv")
df_raw = spark.read \
    .option("header", "true") \
    .option("delimiter", ";") \
    .csv("Files/Bronze/Ventas/anio=2026/pescafresca_2026.csv")
log(f"Extracción completada: {df_raw.count()} filas leídas")

StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] [TooManyRequestsForCapacity] HTTP Response code 430: This Spark job can’t be run because you’ve hit a Spark compute or API rate limit. To proceed, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. For more visibility and control, go to Workspace settings → Job management (Job Concurrency & Queue Monitoring) to review running and queued Spark jobs, understand capacity contention, and take action as needed. [Learn more at 'https://go.microsoft.com/fwlink/?linkid=2356970&clcid=0x409']. HTTP status code: 430.

## Vista previa

In [3]:
#display(df_raw.limit(10))

StatementMeta(, b16cb6f4-2260-4943-bb2a-24cd654d49ad, 5, Finished, Available, Finished, False)

In [3]:
df_raw.printSchema()
log("Schema raw printado")

StatementMeta(, 9c7f8775-6649-4ea9-95e6-2890375e3bbd, 5, Finished, Available, Finished, False)

root
 |-- data: string (nullable = true)
 |-- provincia: string (nullable = true)
 |-- zona: string (nullable = true)
 |-- lonxa: string (nullable = true)
 |-- grupobiologico: string (nullable = true)
 |-- especie: string (nullable = true)
 |-- cantidad: string (nullable = true)
 |-- importe: string (nullable = true)
 |-- unnamed:_8: string (nullable = true)
 |-- anio_archivo: string (nullable = true)



### Valores de nulos

In [3]:
log("Casting de tipos: cantidad/importe a float, cálculo de precio, fecha")

df_dinero = df_raw \
    .withColumn("cantidad", regexp_replace(col("cantidad"), ",", ".").cast(FloatType())) \
    .withColumn("importe", regexp_replace(col("importe"), ",", ".").cast(FloatType()))

df_dinero_v2 =df_dinero.withColumn("precio", when(
    col("cantidad") > 0, 
    round(col("importe") / col("cantidad"), 2)
).otherwise(lit(0.0).cast(FloatType())))

df_fecha = df_dinero_v2.withColumn("data", to_date(col("data"), "yyyy-MM-dd"))
df_fecha.printSchema()
log("Casting completado")


StatementMeta(, 8e69ac39-ecb8-44f6-9ead-2c873dbac286, 5, Finished, Available, Finished, False)

root
 |-- data: date (nullable = true)
 |-- provincia: string (nullable = true)
 |-- zona: string (nullable = true)
 |-- lonxa: string (nullable = true)
 |-- grupobiologico: string (nullable = true)
 |-- especie: string (nullable = true)
 |-- cantidad: float (nullable = true)
 |-- importe: float (nullable = true)
 |-- unnamed:_8: string (nullable = true)
 |-- anio_archivo: string (nullable = true)
 |-- precio: double (nullable = true)



# Creacion del diccionario FAO

##### Leemos la tabla de silver en vez de repetir codigo (DRY) 

In [4]:
log("Lectura de la tabla histórica 'ventas_silver'")

df_historico = spark.read.format("delta").table("ventas_silver")

log(f"Histórico cargado: {df_historico.count()} filas")

StatementMeta(, 8e69ac39-ecb8-44f6-9ead-2c873dbac286, 6, Finished, Available, Finished, False)

In [6]:
#(df_historico.limit(5))

StatementMeta(, b16cb6f4-2260-4943-bb2a-24cd654d49ad, 8, Finished, Available, Finished, False)

### Numero total de diccionario del df_historico de "grupobiologico", "especie", "fao"


In [5]:
df_unicos = df_historico.select("grupobiologico", "especie", "fao").distinct()

filas = df_unicos.collect()

diccionario_fao = {
    (fila["grupobiologico"], fila["especie"]): fila["fao"]
    for fila in filas
}

log(f"\n El numero totales de grupobiologico-espcie = fao son: {len(diccionario_fao)}")

StatementMeta(, 8e69ac39-ecb8-44f6-9ead-2c873dbac286, 7, Finished, Available, Finished, False)


 El numero totales de grupobiologico-espcie = fao son: 352


#### Correccion ya de las tablas con la forma columnar final e implementacion de la fao

In [8]:
log("Aplicando mapeo FAO (primera pasada)")
mapeo_fao = when(col("especie") == "inicial", "AAA")
for(grupo, especie), codigo_fao in diccionario_fao.items():
    mapeo_fao = mapeo_fao.when((col("grupobiologico") == grupo) & (col("especie") == especie), codigo_fao)

mapeo_fao = mapeo_fao.otherwise(lit("Desconocido"))

df_silver_2026 = df_fecha.withColumn("fao", mapeo_fao)
df_silver_2026 = df_silver_2026.withColumn("anio_archivo", lit(2026).cast(IntegerType()))

columnas_target = [
    "data", "grupobiologico", "fao", "especie", "provincia", 
    "zona", "lonxa", "cantidad", "importe", "precio", "anio_archivo"
]
df_2026_final = df_silver_2026.select(*columnas_target)

df_2026_final.printSchema()
log("Mapeo FAO inicial aplicado, esquema columnar final seleccionado")

StatementMeta(, b16cb6f4-2260-4943-bb2a-24cd654d49ad, 10, Finished, Available, Finished, False)

root
 |-- data: date (nullable = true)
 |-- grupobiologico: string (nullable = true)
 |-- fao: string (nullable = false)
 |-- especie: string (nullable = true)
 |-- provincia: string (nullable = true)
 |-- zona: string (nullable = true)
 |-- lonxa: string (nullable = true)
 |-- cantidad: float (nullable = true)
 |-- importe: float (nullable = true)
 |-- precio: double (nullable = true)
 |-- anio_archivo: integer (nullable = false)



In [12]:
#display(df_2026_final.limit(10))

StatementMeta(, 1b51c90c-58ae-4c91-b3cc-79ffaefad509, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9b308ca1-d3ed-4eaf-9b82-53c841413121)

# Comprobacion de los lits

### Sabemos que el grupobiologico estan bien, el problema son las especies
### Tomamos como diccionario el df_historico para corregir los nombres de las espicies


##### Contrastado con url: https://deondesenon.xunta.gal/es/productos-de-la-marca/que-productos-me-ofrece-la-marca/peces

In [9]:
escpie_formateo = df_2026_final.filter(col("fao") == "Desconocido").select("especie").distinct().collect()

lista_especies = [fila["especie"] for fila in escpie_formateo]

print(f"\n Las espececies con fao desconocido: \n{lista_especies}")
log(f"Especies con FAO desconocido tras 1ª pasada: {len(lista_especies)}")

StatementMeta(, b16cb6f4-2260-4943-bb2a-24cd654d49ad, 11, Finished, Available, Finished, False)


 Las espececies con fao desconocido: 
['Escolar negro o pez mantequilla', 'Raia', 'Fodon', 'Relo/reloxo', 'Ameixon', 'Volador', 'Boi bravo', 'Raya santiaguesa', 'Peixe-espada', 'Raya boca de rosa', 'Escacho', 'Salmonete', 'Bonito do atlántico', 'Alga verde', 'Alga roja', 'Bodión', 'Rapapelos de kessler', 'Rapapelo de ollos grandes', 'Lorcho', 'Alosa', 'Cherna', 'Merlan', 'Anguía', 'Gallineta', 'Mero branco', 'Gallano', 'Pintarroja bocanegra']


# Armonizacion de datos
## Miramos los "grupos biologicos y "espiecie" si existen en los datos historicos
##### si "grupo biologicos" es el error se marca con || con el nuevo gruobiologico
##### Si la "espiece" es el error se marca con = con el nueva especie

Observaciones:
- Relo/reloxo = Reló
- Raia = Raias 
- Raya boca de rosa = Raia boca de rosa
- Raya santiaguesa = Raia santiaguesa
- Fodon = Fodón
- Escolar negro o pez mantequilla = Escolar negro
- Ameixon = Ameixa fina
- Volador = Pota voadora
- Boi bravo = Boi
- Peixe-espada = Peixe espada
- Escacho = Escachos
- Salmonete = Salmonetes
- Bonito do atlantico = Bonito do Atlántico
- Alga verde  || Algas
- Alga roja  || Algas
- Bodión = Serrán (xddddddd)
- Rapapelos de kessler = Rapapelos de Kessler
- Chera = Cherna de ley
- Rapapelo de ollos grandes = Rapapelos de ollos grandes
- Lorcho = Lorchos
- Alosa = Saboga
- Merlan = Merlán
- Anguía = Anguilas
- Gallineta = Cabras (xdddddd) 
- Mero branco = Mero
- Gallano = Rei
- Lisa No hay historial hasta 2026, se empieza a pescar ahora. Fao: MLR grupo biologicos:Peixes
- Pintarroja bocanegra = Melgacho
- Ochavo = Sin uso comercial sin Fao se puede eliminar se puede hacer un .drop()


In [10]:
log("Inicio de armonización: corrigiendo nombres de especies y grupos")

correcciones_especies = {
    "Relo/reloxo": "Reló",
    "Raia": "Raias",
    "Raya boca de rosa": "Raia boca de rosa",
    "Raya santiaguesa": "Raia santiaguesa",
    "Fodon": "Fodón",
    "Ameixon": "Ameixa fina",
    "Volador": "Pota voadora",
    "Boi bravo": "Boi",
    "Peixe-espada": "Peixe espada",
    "Escacho": "Escachos",
    "Salmonete": "Salmonetes",
    "Bonito do atlantico": "Bonito do Atlántico",
    "Bonito do atlántico": "Bonito do Atlántico",
    "Bodión": "Serrán",
    "Rapapelos de kessler": "Rapapelos de Kessler",
    "Cherna": "Cherna de ley",
    "Rapapelo de ollos grandes": "Rapapelos de ollos grandes",
    "Lorcho": "Lorchos",
    "Alosa": "Saboga",
    "Merlan": "Merlán",
    "Anguía": "Anguilas",
    "Gallineta": "Cabras",
    "Mero branco": "Mero",
    "Gallano": "Rei",
    "Pintarroja bocanegra": "Melgacho",
    "Escolar negro o pez mantequilla" : "Escolar negro",
    "Alga verde" : "Algas verdes",
    "Alga roja": "Algas vermellas"
}

correcciones_grupo ={
    "Alga verde" : "Algas",
    "Alga roja" : "Algas"
}
###### Se debe cambiar para agilizar el computo, funciona bien con pocas especies, pero el motor Catalist sufria con mas carga incremental a medida con el tiempo
#df_2026_final = df_2026_final\
 #   .replace(correcciones_especies, subset=["especie"])\
  #  .replace(correcciones_grupo, subset=["grupobiologico"])
###################

datos_especies = list(correcciones_especies.items())
datos_grupo = list(correcciones_grupo.items())

columnas_temporales = ["especie_sucia", "especie_limpia", "grupo_sucio", "grupo_limpio"]
df_2026_final = df_2026_final.drop(*columnas_temporales)

#Se crean tablas dimensionales con esas especificaciones en memoria, es poca. Pasa de python a Spark
df_dic_especies = spark.createDataFrame(datos_especies, ["especie_sucia", "especie_limpia"])
df_dic_grupo = spark.createDataFrame(datos_grupo, ["grupo_sucio", "grupo_limpio"])

## Se cruzan los datos con la tabla principal

df_cruzado_2026 = df_2026_final.alias("main").join(
    broadcast(df_dic_especies).alias("dic_esp"),
    col("main.especie") == col("dic_esp.especie_sucia"),
    how="left"
).join(
    broadcast(df_dic_grupo).alias("dic_gru"),
    col("main.grupobiologico") == col("dic_gru.grupo_sucio"),
    how="left"
)

### Uso de coalesce() metodo que lle de izquierda a derecha y se queda con el primer valor no nulo.
###Si la espcie cruzo con el diccionario, usa "especie_limpia". Si no cruzo (es nulo) y mantiene la que tenia de especie

df_2026_final = df_cruzado_2026.withColumn(
    "especie", coalesce(col("dic_esp.especie_limpia"), col("main.especie"))
).withColumn(
    "grupobiologico", coalesce(col("dic_gru.grupo_limpio"), col("main.grupobiologico"))
)

df_2026_final = df_2026_final.drop(*columnas_temporales)

#display(df_2026_final.limit(10))
log(f"Armonización completada: {len(correcciones_especies)} correcciones de especie, {len(correcciones_grupo)} de grupo")


StatementMeta(, b16cb6f4-2260-4943-bb2a-24cd654d49ad, 12, Finished, Available, Finished, False)

# Adjudicacion del Fao

In [11]:
#### Se añade nuevas especies al diccionario por si se incrementa.
diccionario_fao[("Peixes", "Lisa")] = "MLR"
diccionario_fao[("Peixes", "Ochavo")] = "Desperdicio"
#######
log(f"Diccionario FAO ampliado a {len(diccionario_fao)} entradas (añadidas Lisa y Ochavo)")
log("Aplicando mapeo FAO (segunda pasada tras armonización)")

mapeo_fao = when(col("especie") == "inicial", "AAA")

for(grupo, especie), codigo_fao in diccionario_fao.items():
    mapeo_fao = mapeo_fao.when((col("grupobiologico") == grupo) & (col("especie") == especie), codigo_fao)

mapeo_fao = mapeo_fao.otherwise(lit("Desconocido"))

df_silver_2026 = df_2026_final.withColumn("fao", mapeo_fao)
log("Segundo mapeo FAO aplicado")

StatementMeta(, b16cb6f4-2260-4943-bb2a-24cd654d49ad, 13, Finished, Available, Finished, False)

## Comprobacion de Fao, Especie y grupoBiologico

In [12]:
escpie_formateo = df_silver_2026.filter(col("fao") == "Desconocido").select("especie").distinct().collect()

lista_especies = [fila["especie"] for fila in escpie_formateo]

print(f"\nLa lista de Desconocidos en la columna fao es: {len(lista_especies)}")
log(f"Especies con FAO desconocido tras 2ª pasada: {len(lista_especies)} -> {lista_especies}")


StatementMeta(, b16cb6f4-2260-4943-bb2a-24cd654d49ad, 14, Finished, Available, Finished, False)


La lista de Desconocidos en la columna fao es: 0


## Que hacer con los precios = 0.

In [13]:
df_prueba_precio = df_silver_2026.filter(col("precio") == 0.0)
log(f"Filas con precio = 0 detectadas: {df_prueba_precio.count()}")


display(df_prueba_precio.limit(10))

StatementMeta(, b16cb6f4-2260-4943-bb2a-24cd654d49ad, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ade5c38e-c6c2-4844-86c0-cb7c0508949d)

### Como se ve tenemos columnas en las que el "importe" y "precio" son cero:

#### Metodo para compensar:
- Calcular la mediana del precio historico por cada codigo fao = diccionario_precios.
- Cruce de información "diccionario_precios" con los datos de 2026.
- Si el importe es 0.0 se calcula : df_silver_2026.filter(col("cantidad")) * diccionario_precio.

In [14]:
log("Imputación de precios: calculando mediana mensual por FAO desde el histórico")

## Extraemos los meses de df_historico y de df_2026.

df_historico_mes = df_historico.withColumn("mes_venta", month(col("data")))
df_silver_2026_mes = df_silver_2026.withColumn("mes_venta", month(col("data")))

#Obtenemos los 2 percentiles de cada fao de los historicos, y adaptarnos a las temporadas.
diccionario_precio = df_historico_mes.filter(col("precio") > 0)\
    .groupBy("fao", "mes_venta")\
    .agg(expr("percentile_approx(precio, 0.5)").alias("Precio_medio_Fao_Mes"))

log(f"Diccionario de precio mediano FAO-mes: {diccionario_precio.count()} combinaciones")

#Cruzamos los datos, ya tiene la columnas de "Precio_medio_Fao_Mes"
df_importe = df_silver_2026_mes.join(
    broadcast(diccionario_precio),
    on=["fao","mes_venta"], #exige un cruce con las 2 columnas creadas
    how="left"
)

df_silver_2026 = df_importe.withColumn(
    "importe", when(col("importe")== 0.0, col("cantidad") * col("Precio_medio_Fao_Mes")).otherwise(col("importe"))
).withColumn(
    "precio", when(col("precio") == 0.0, col("Precio_medio_Fao_Mes")).otherwise(col("precio"))
)

df_silver_2026 = df_silver_2026.drop("Precio_medio_Fao_Mes", "mes_venta")
log("Imputación de precios e importes completada")

StatementMeta(, b16cb6f4-2260-4943-bb2a-24cd654d49ad, 16, Finished, Available, Finished, False)

#### Comprobacion, todo en orden 

In [16]:
df_fao = df_silver_2026.filter(col("importe").isNull())
log(f"Filas con importe nulo tras imputación: {df_fao.count()}")


#display(df_fao.limit(10))

### Estos son los ochavo que no se comercializan pero es que ouede ser algo mas son 6 registros

StatementMeta(, b16cb6f4-2260-4943-bb2a-24cd654d49ad, 18, Finished, Available, Finished, False)

#### Ultimo registo para saber si existe nulos.

In [17]:
df_silver_2026.select([
    count(when(isnull(c), c)).alias(c) for c in df_silver_2026.columns
]).show()
log(f"Conteo final de nulos: {_nulls_df.collect()[0].asDict()}")

StatementMeta(, b16cb6f4-2260-4943-bb2a-24cd654d49ad, 19, Finished, Available, Finished, False)

+---+----+--------------+-------+---------+----+-----+--------+-------+------+------------+
|fao|data|grupobiologico|especie|provincia|zona|lonxa|cantidad|importe|precio|anio_archivo|
+---+----+--------------+-------+---------+----+-----+--------+-------+------+------------+
|  0|   0|             0|      0|        0|   0|    0|       0|      6|     6|           0|
+---+----+--------------+-------+---------+----+-----+--------+-------+------+------------+



In [18]:
columnas_target = [
    "data", "grupobiologico", "fao", "especie", "provincia", 
    "zona", "lonxa", "cantidad", "importe", "precio", "anio_archivo"
]
df_silver_2026 = df_silver_2026.select(*columnas_target)
log(f"Selección de columnas final: {columnas_target}")

StatementMeta(, b16cb6f4-2260-4943-bb2a-24cd654d49ad, 20, Finished, Available, Finished, False)

In [19]:
df_silver_2026 = df_silver_2026.withColumn("precio", col("precio").cast(FloatType()))
log("Cast final de 'precio' a FloatType")

StatementMeta(, b16cb6f4-2260-4943-bb2a-24cd654d49ad, 21, Finished, Available, Finished, False)

In [20]:
df_silver_2026.printSchema()

StatementMeta(, b16cb6f4-2260-4943-bb2a-24cd654d49ad, 22, Finished, Available, Finished, False)

root
 |-- data: date (nullable = true)
 |-- grupobiologico: string (nullable = true)
 |-- fao: string (nullable = false)
 |-- especie: string (nullable = true)
 |-- provincia: string (nullable = true)
 |-- zona: string (nullable = true)
 |-- lonxa: string (nullable = true)
 |-- cantidad: float (nullable = true)
 |-- importe: float (nullable = true)
 |-- precio: float (nullable = true)
 |-- anio_archivo: integer (nullable = false)



# Hacemos un appendp para las tablas delta parquet-

In [21]:
# 4. CARGA SE SEGURIDAD (Append)
# Como las columnas son idénticas en nombre y tipo, se apilarán perfectamente en OneLake bajo la partición anio_archivo=2026
df_silver_2026.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("ventas_silver")
log("FIN: APPEND completado en la tabla 'ventas_silver'")

StatementMeta(, b16cb6f4-2260-4943-bb2a-24cd654d49ad, 23, Finished, Available, Finished, False)

### Comprobacion de faos totales y aprovechamos para crear la dim_fao para aprovechar en gold

In [8]:
df_dim_pescadito = df_historico.select("grupobiologico", "especie", "fao").distinct()

df_dim_pescadito = df_dim_pescadito.dropna(subset=["fao", "especie"])

df_dim_pescadito.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_especies")

print(f"Dimension en OneLake. Filas {df_dim_pescadito.count()} ")

StatementMeta(, 8e69ac39-ecb8-44f6-9ead-2c873dbac286, 10, Finished, Available, Finished, False)

Dimension en OneLake. Filas 354 


In [9]:
from notebookutils import mssparkutils

# Esto sí apaga el clúster físico desde el código
mssparkutils.session.stop()

StatementMeta(, 8e69ac39-ecb8-44f6-9ead-2c873dbac286, 11, Finished, Available, Finished, True)